# Analyze and save Raman spectra

In [ ]:
import io
from pathlib import Path

import arcadia_pycolor as apc
import numpy as np
import pandas as pd
import plotly.colors as pcolors
import plotly.express as px
import plotly.graph_objects as go
import ramanspy as rp
from PIL import Image
from plotly.subplots import make_subplots
from scipy.optimize import curve_fit

from raman_analysis.utils import collect_spectra_from_dir, plot_spectra, plot_spectra_in_dir

apc.plotly.setup()
%load_ext autoreload
%autoreload 2

repo_root = Path().resolve().parents[0]
outputs_dir = repo_root / "outputs"
data_dir = repo_root / "data"
raman_chem_perturbation_dir = data_dir / "raman-chemical-perturbations"
raman_ph_perturbation_dir = data_dir / "raman-ph-perturbations"
plot_dir_htmls = outputs_dir / "Raman" / "html"
plot_dir_png = outputs_dir / "Raman" / "png"



## RamanSpy processing pipeline

In [ ]:
rp_pipeline = rp.preprocessing.Pipeline(
    [
        rp.preprocessing.misc.Cropper(region=(350, 1800)),
        rp.preprocessing.despike.WhitakerHayes(),
        rp.preprocessing.denoise.SavGol(window_length=5, polyorder=3),
        rp.preprocessing.baseline.ModPoly(poly_order=9),
    ]
)

## Chemical perturbations


### Colormap


In [ ]:
chem_perturbation_palette = apc.Palette(
    "chem_perturbations", [apc.black, apc.vital, apc.aster, apc.lime]
)
chem_perturbation_palette

In [ ]:
def generate_subtract_dict_chem(samples):
    """Generate a dictionary mapping sample names to their buffer names for subtraction."""
    subtract_dict = {}
    for name in samples:
        date = name.split("/")[0]
        buffer = name.split(" in ")[-1]
        subtract_dict[name] = f"{date}/{buffer}"
    return subtract_dict

### Load data

In [ ]:
data_list, sample_df = collect_spectra_from_dir(raman_chem_perturbation_dir, "instantraman", recursive=True)



### Utility function to print sample table

In [ ]:
plotted_samples = {} # Track all plotted samples for raw data
df_sample_tables = []
def print_sample_table(df, subtract_dict):
    proteins = []
    buffers = []
    for k,v in subtract_dict.items():
        protein_filename = df.index[df['Name'] == k].tolist()[0]
        proteins.append(protein_filename)
        if v is None:
            buffers.append(None)
            continue
        buffer_filename = df.index[df['Name'] == v].tolist()[0]
        buffers.append(buffer_filename)
    df = pd.DataFrame({
        "Protein experiment": list(subtract_dict.keys()),
        "Protein filename": proteins,
        "Buffer filename": buffers
    })
    return df


### Utility functions to normalize spectra to the Phe peak

In [ ]:
Phe_wavenumber = 1003 # Phenylalanine peak, used for normalization


def peak_max(x, y, center, window=5):
    """Max of y within ±window cm⁻¹ of center (the peak max can fall between spectral points,
    so a window is more robust than the single nearest point)."""
    x, y = np.asarray(x), np.asarray(y)
    return y[np.abs(x - center) <= window].max()


def normalize_to_peak(fig, center, window=5):
    """Divide each trace by its max within ±window cm⁻¹ of center."""
    for trace in fig.data:
        trace.y = np.asarray(trace.y) / peak_max(trace.x, trace.y, center, window)


def normalize_to_phe(fig, window=5):
    """Divide each trace by its max within ±window cm⁻¹ of the Phe peak."""
    normalize_to_peak(fig, Phe_wavenumber, window)


def yrange_in_xrange(fig, xrange, pad=0.05):
    """y-axis range spanning all traces within xrange, with fractional padding."""
    y_in_range = np.concatenate([
        np.asarray(trace.y)[(np.asarray(trace.x) >= xrange[0]) & (np.asarray(trace.x) <= xrange[1])]
        for trace in fig.data
    ])
    y_min, y_max = y_in_range.min(), y_in_range.max()
    margin = pad * (y_max - y_min)
    return [y_min - margin, y_max + margin]

### All available proteins

In [ ]:

buffers = ["Tris", "Tris+TCEP", "Tris+EDTA", "Tris+TCEP+EDTA"]
all_samples = sample_df["Name"].str.split("/").str[-1].str.split(" in ").str[0].unique()
all_samples = [s for s in all_samples if not any(b in s for b in buffers)]
all_samples

### Utility function to write progressive PNGs (for slides)

In [ ]:
def write_progressive_pngs(fig, buffers, out_dir, stem):
    """Write one PNG per buffer condition, cumulatively adding traces (Tris, then Tris + Tris+TCEP, ...).
    Axis ranges are frozen to those of the full figure so all PNGs share the same axes."""
    full_fig = fig.full_figure_for_development(warn=False)
    fig.update_xaxes(range=full_fig.layout.xaxis.range)
    fig.update_yaxes(range=full_fig.layout.yaxis.range)

    trace_buffers = [trace.name.split(" in ")[-1] for trace in fig.data]
    present_buffers = [buf for buf in buffers if buf in trace_buffers]
    out_dir.mkdir(parents=True, exist_ok=True)
    for k in range(len(present_buffers)):
        shown_buffers = present_buffers[: k + 1]
        step_fig = go.Figure(fig)
        for trace, buf in zip(step_fig.data, trace_buffers, strict=True):
            trace.visible = buf in shown_buffers
        step_fig.write_image(out_dir / f"{stem}_{k + 1}_{shown_buffers[-1]}.png", scale=3)

### Plot spectra for BSA, FAF-BSA, DNAse, Cytochrome C

In [ ]:
xranges = {
    "BSA [10 mgml]": [None, [360, 700], [1150, 1378]],
    "FAF-BSA [10 mgml]": [None, [360, 700], [1150, 1378]],
    "DNAse [12.5 mgml]": [None, [360, 700], [1150, 1378]],
    "CytC [10 mgml]": [None, [360, 700], [1500, 1650]],
}
# y ranges in Phe-normalized units; None = auto (zooms fit the data within the x range)
yranges = {
    "BSA [10 mgml]": [None, None, None],
    "FAF-BSA [10 mgml]": [None, None, None],
    "DNAse [12.5 mgml]": [None, None, None],
    "CytC [10 mgml]": [None, None, None],
}

# First plot is full spectrum, the second and third are zooms
fig_widths = ["full_wide", "half_square", "half_square"]

n_groups = sample_df["Group ID"].nunique()

buffer_dict = {
    "BSA [10 mgml]": buffers[:-1], # TCEP+EDTA excluded — causes precipitation in BSA
    "FAF-BSA [10 mgml]": buffers[:-1], # TCEP+EDTA excluded — causes precipitation in BSA
    "DNAse [12.5 mgml]": buffers,
    "CytC [10 mgml]": buffers,
}


for protein in xranges.keys():
    for i, (xrange, yrange) in enumerate(zip(xranges[protein], yranges[protein], strict=True)):
        # include files that are the protein in the corresponding buffer
        include_samples = [
            n
            for n in sample_df["Name"]
            if Path(n).name.startswith(protein)
            and any(n.endswith(buf) for buf in buffer_dict[protein])
        ]
        subtract_id = generate_subtract_dict_chem(include_samples)
        include_samples += list(subtract_id.values())
        fig, names, rp_container = plot_spectra_in_dir(
            data_list,
            col_to_plot="Intensity (dark-subtracted)",
            data_dir=raman_chem_perturbation_dir,
            subtract_id=subtract_id,
            rp_pipeline=rp_pipeline,
            colors=chem_perturbation_palette,
            include=include_samples,
            xaxis_title="Raman shift (cm⁻¹)",
            yaxis_title="Intensity (norm. to Phe)",
        )
        fig.data = tuple(trace for trace in fig.data if trace.name not in subtract_id.values())
        normalize_to_phe(fig)
        if yrange is None and xrange is not None:
            yrange = yrange_in_xrange(fig, xrange)
        fig.update_xaxes(range=xrange)
        fig.update_yaxes(range=yrange)
        apc.plotly.style_plot(fig, monospaced_axes="all")
        apc.plotly.set_figure_dimensions(fig, size=fig_widths[i])
        fig.update_layout(showlegend=False, title_text=None)
        protein_dir_png = plot_dir_png / protein
        protein_dir_html = plot_dir_htmls / protein
        protein_dir_png.mkdir(parents=True, exist_ok=True)
        protein_dir_html.mkdir(parents=True, exist_ok=True)

        write_progressive_pngs(
            fig, buffer_dict[protein], protein_dir_png / "progressive", f"{protein}_{xrange}_cm-1"
        )
        if xrange is not None:  # zoomed insets: also save a single plot with all conditions
            fig.write_image(protein_dir_png / f"{protein}_{xrange}_cm-1.png", scale=3)
        fig.write_html(protein_dir_html / f"{protein}_{xrange}_cm-1.html")
        fig.show()
    df_sample_tables.append(print_sample_table(sample_df, subtract_id))

### Plot spectra for additional proteins (not in main figures)

In [ ]:
additional_proteins = [s for s in all_samples if s not in xranges.keys()]
for protein in additional_proteins:
    include_samples = [
        n
        for n in sample_df["Name"]
        if Path(n).name.startswith(protein)
        and any(n.endswith(buf) for buf in buffers)
    ]
    subtract_id = generate_subtract_dict_chem(include_samples)
    include_samples += list(subtract_id.values())
    fig, names, rp_container = plot_spectra_in_dir(
        data_list,
        col_to_plot="Intensity (dark-subtracted)",
        data_dir=raman_chem_perturbation_dir,
        subtract_id=subtract_id,
        rp_pipeline=rp_pipeline,
        colors=chem_perturbation_palette,
        include=include_samples,
        xaxis_title="Raman shift (cm⁻¹)",
        yaxis_title="Intensity (norm. to Phe)",
    )

    df_sample_tables.append(print_sample_table(sample_df, subtract_id))

    fig.data = tuple(trace for trace in fig.data if trace.name not in subtract_id.values())
    normalize_to_phe(fig)

    apc.plotly.style_plot(fig, monospaced_axes="all")
    apc.plotly.set_figure_dimensions(fig, size="full_wide")
    fig.update_layout(showlegend=False, title_text=None)

    protein_dir_png = plot_dir_png / protein
    protein_dir_html = plot_dir_htmls / protein
    protein_dir_png.mkdir(parents=True, exist_ok=True)
    protein_dir_html.mkdir(parents=True, exist_ok=True)

    write_progressive_pngs(fig, buffers, protein_dir_png / "progressive", f"{protein}_None_cm-1")
    fig.write_html(protein_dir_html / f"{protein}_None_cm-1.html")
    fig.show()

### Plot spectra for buffers

In [ ]:
buffers

In [ ]:
include_samples = [n for n in sample_df["Name"] if any(n == f"2025-12-10/{buf}" for buf in buffers)]

fig, names, rp_container = plot_spectra_in_dir(
    data_list,
    col_to_plot="Intensity (dark-subtracted)",
    data_dir=raman_chem_perturbation_dir,
    subtract_id=None,
    rp_pipeline=rp_pipeline,
    colors=chem_perturbation_palette,
    include=include_samples,
    xaxis_title="Raman shift (cm⁻¹)",
    yaxis_title="Intensity (a.u.)",
    title="Buffers",
)

subtract_id = {k:None for k in include_samples}
df_sample_tables.append(print_sample_table(sample_df, subtract_id))

apc.plotly.style_plot(fig, monospaced_axes="all")
apc.plotly.set_figure_dimensions(fig, size="full_wide")

fig.write_image(plot_dir_png / "buffers.png", scale=3)
fig.write_html(plot_dir_htmls / "buffers.html")
fig.show()

## pH perturbations

### Colormap and constants

In [ ]:
ph_fig_width = 1300
ph_fig_height = 400
ph_gradient = apc.gradients.red_blue
BSA_color = apc.aster
BSA_TCEP_color = apc.amber

ph_to_plot = [0.3, 1, 1.3, 2, 2.3, 3, 7.5, 8, 9, 9.5, 12, 12.7]
ph_traces_to_show = ph_to_plot # modify to show only a subset of traces
ph_vals_to_show_on_colorbar = [0.3, 7.5, 12.7]

### Load data

In [ ]:
data_list, sample_df = collect_spectra_from_dir(
    raman_ph_perturbation_dir, "instantraman", recursive=True
)

n_groups = sample_df["Group ID"].nunique()

In [ ]:
def generate_subtract_dict_pH(_replicate, sample_string):
    """ Make dictionary mapping sample names to pH-matched buffer names. """
    ph_vals = [0, 0.3, 1, 1.3, 2, 2.3, 3, 3.3, 4, 5, 6,
                7.5, 8, 9, 9.5, 10, 10.7, 11, 11.7, 12,
                12.7, 13, 13.7, 14]
    ph_vals_str = [str(p) for p in ph_vals]

    sample_names = sample_df["Name"].unique().tolist()
    all_protein = [n for n in sample_names if sample_string + " pH" in n and _replicate in n]

    subtraction_key = "TCEP" if "TCEP" in sample_string else "buffers"
    subtract_dict = {}
    for p in all_protein:
        for ph in ph_vals_str:
            if f"{sample_string} pH {ph}" in p:
                subtract_dict[p] = [
                    b for b in sample_names if b.endswith(f"{_replicate}/{subtraction_key} pH {ph}")
                ][0]
    # For non-TCEP data, subtract pH=7.5 data instead of matching pH to account for
    # peak shifts in the buffer solutions that are not seen in the protein solutions
    for k, v in list(subtract_dict.items()):
        if "TCEP" not in v:
            if any(v.endswith(f"pH {ph}") for ph in ["8", "9", "9.5"]):
                new_v = v.rsplit(" ", 1)[0] + " 7.5"
                subtract_dict[k] = new_v
    return subtract_dict

### Utility function to write looping pH animation (for slides)

In [ ]:
def write_ph_gif(fig, ph_vals, out_path, frame_ms=800, scale=2):
    """Write an infinitely looping GIF showing one pH trace per frame, in the order of ph_vals.
    Axis ranges are frozen to those of the full figure so all frames share the same axes."""
    full_fig = fig.full_figure_for_development(warn=False)
    fig = go.Figure(fig)
    fig.update_xaxes(range=full_fig.layout.xaxis.range)
    fig.update_yaxes(range=full_fig.layout.yaxis.range)
    fig.update_layout(showlegend=False, title_text=None)

    frames = []
    for ph in ph_vals:
        for trace in fig.data:
            if trace.name is not None:  # colorbar trace has no name
                trace.visible = trace.name.rsplit("pH ", 1)[-1] == str(ph)
        fig.layout.annotations = []
        fig.add_annotation(
            text=f"pH {ph}",
            x=0.01,
            y=0.98,
            xref="paper",
            yref="paper",
            xanchor="left",
            yanchor="top",
            showarrow=False,
            font=dict(size=16),
        )
        frame = Image.open(io.BytesIO(fig.to_image(format="png", scale=scale)))
        frames.append(frame.convert("RGB").quantize(colors=256, method=Image.Quantize.MEDIANCUT))

    frames[0].save(
        out_path, save_all=True, append_images=frames[1:], duration=frame_ms, loop=0, optimize=False
    )

### Plot Raman spectra of BSA under different pH conditions

In [ ]:
rp_pipeline = rp.preprocessing.Pipeline(
    [
        rp.preprocessing.misc.Cropper(region=(350, 1800)),
        rp.preprocessing.despike.WhitakerHayes(),
        rp.preprocessing.denoise.SavGol(window_length=5, polyorder=3),
        # rp.preprocessing.baseline.ModPoly(poly_order=3),
        rp.preprocessing.baseline.IASLS(lam=1e2)
    ]
)

In [ ]:
replicate = "rep1" # replicate ID to plot
ph_to_plot_labels = [f"pH {ph_val}" for ph_val in ph_to_plot]

spectral_regions = {
    (1400, 1420): "Carboxylate",
    (1647, 1657): "Amide I<br>⍺-Helix",
    (1328, 1341): "Trp,<br>C⍺-H def",
    (940, 950): "C⍺-C stretch",
}

sample_names_dict = {}
sample_containers_dict = {}

for sample in ["BSA", "BSA+TCEP"]:
    subtract_id = generate_subtract_dict_pH(replicate, sample)
    subtract_id = {
        k: v
        for k, v in subtract_id.items()
        if any(k.endswith(label) for label in ph_to_plot_labels)
    }

    include_samples = list(subtract_id.keys()) + list(subtract_id.values())
    exclude_from_plot = subtract_id.values()

    colors = pcolors.sample_colorscale(px.colors.qualitative.Dark24,n_groups)

    fig, names, rp_container = plot_spectra_in_dir(
        data_list,
        col_to_plot="Intensity (dark-subtracted)",
        data_dir=raman_ph_perturbation_dir,
        subtract_id=subtract_id,
        rp_pipeline=rp_pipeline,
        colors=colors,
        include=include_samples,
        height=ph_fig_height,
        width=ph_fig_width,
        xaxis_title="Raman shift (cm⁻¹)",
        yaxis_title="Intensity (norm. to Phe)",
        title=f"{sample} pH perturbation",
    )
    fig.data = tuple(
        trace
        for trace in fig.data
        if not any(trace.name == exclude for exclude in exclude_from_plot)
    )
    normalize_to_phe(fig)
    for trace in fig.data:
        if not any(str(trace.name).endswith(f"pH {ph}") for ph in ph_traces_to_show):
            trace.visible = "legendonly"

    gradient = ph_gradient.resample_as_palette(len(fig.data))
    for i, trace in enumerate(fig.data):
        trace.line.color = gradient[i]

    # Add a colorbar for the gradient with pH values
    colorbar_trace = go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale=[[i / (len(gradient) - 1), color] for i, color in enumerate(gradient)],
            cmin=min(ph_to_plot),
            cmax=max(ph_to_plot),
            colorbar=dict(
                title="pH",
                tickvals=ph_to_plot,
                ticktext=[str(ph) for ph in ph_to_plot],
                lenmode="pixels",
                len=200,
            ),
            color=[min(ph_to_plot), max(ph_to_plot)],
            showscale=True,
            size=0.1,
        ),
        showlegend=False,
        hoverinfo="none",
    )
    # pH color bar
    colorbar_trace.marker.colorbar.update(
        x=1.0,
        y=1.05,
        xanchor="left",
        yanchor="bottom",
        orientation="h",
        thickness=5,
        lenmode="pixels",
        len=200,
        title_side="top",
        tickvals=ph_vals_to_show_on_colorbar,
        ticktext=[str(ph) for ph in ph_vals_to_show_on_colorbar],
    )
    fig.add_trace(colorbar_trace)


    sample_names_dict[sample] = names
    sample_containers_dict[sample] = rp_container
    df_sample_tables.append(print_sample_table(sample_df, subtract_id))

    fig.update_layout(margin=dict(t=100))
    apc.plotly.style_plot(fig, monospaced_axes="all")
    apc.plotly.set_figure_dimensions(fig, size="full_wide")
    fig.show()

    fig.write_image(plot_dir_png / f"{sample}_ph_spectra.png", scale=3)
    fig.write_html(plot_dir_htmls / f"{sample}_ph_spectra.html")
    write_ph_gif(fig, ph_to_plot, plot_dir_png / f"{sample}_ph_spectra.gif")


### Plot protein features as a function of pH

In [ ]:
def intensity_at(spectral_axis, spectral_data_row, wavenumber):
    idx = np.argmin(np.abs(spectral_axis - wavenumber))
    return float(spectral_data_row[idx])


def fit_gaussian_peak(spectral_axis, spectral_data_row, center, window=60):
    """Fit a Gaussian to a spectral region; return (center, fwhm). Returns (nan, nan) on failure."""
    mask = (spectral_axis >= center - window) & (spectral_axis <= center + window)
    x, y = spectral_axis[mask], spectral_data_row[mask]
    if len(x) < 4:
        return np.nan, np.nan

    def gaussian(x, amp, mu, sigma):
        return amp * np.exp(-((x - mu) ** 2) / (2 * sigma**2))

    try:
        p0 = [y.max(), center, window / 3]
        bounds = ([0, center - window, 1], [np.inf, center + window, window])
        popt, _ = curve_fit(gaussian, x, y, p0=p0, bounds=bounds, maxfev=5000)
        _, mu, sigma = popt
        fwhm = 2.355 * abs(sigma)
        return float(mu), float(fwhm)
    except RuntimeError:
        return np.nan, np.nan


peak_records = []
for sample in ["BSA", "BSA+TCEP"]:
    names = sample_names_dict[sample]
    container = sample_containers_dict[sample]
    ax = container.spectral_axis
    for i, name in enumerate(names):
        if "BSA" not in name:
            continue
        for ph in ph_to_plot:
            if not name.endswith(f"pH {ph}"):
                continue
            row = container.spectral_data[i]
            center_1657, fwhm_1657 = fit_gaussian_peak(ax, row, 1657)
            peak_records.append(
                {
                    "pH": ph,
                    "sample": sample,
                    "i1341": intensity_at(ax, row, 1341) / intensity_at(ax, row, Phe_wavenumber),
                    "i1657": (
                        intensity_at(ax, row, center_1657) / intensity_at(ax, row, Phe_wavenumber)
                    ),
                    "i1657_center": center_1657,
                    "i1657_fwhm": fwhm_1657,
                    "i1412": intensity_at(ax, row, 1412) / intensity_at(ax, row, Phe_wavenumber),
                    "i945": intensity_at(ax, row, 945) / intensity_at(ax, row, Phe_wavenumber),
                }
            )

peak_df = pd.DataFrame(peak_records)

peak_params = [
    ("i1657", "Amide I peak", "Normalized intensity"),
    ("i1657_center", "Amide I center", "Raman shift (cm⁻¹)"),
    ("i1657_fwhm", "Amide I FWHM", "Wavenumber (cm⁻¹)"),
    ("i1341", "Trp, C⍺-H def (1,341 cm⁻¹)", "Normalized intensity"),
    ("i945", "C⍺-C stretch (945 cm⁻¹)", "Normalized intensity"),
    ("i1412", "Carboxylate (1,412 cm⁻¹)", "Normalized intensity"),
]

n_cols = 3
n_rows = int(np.ceil(len(peak_params) / n_cols))
sample_colors = {"BSA": BSA_color, "BSA+TCEP": BSA_TCEP_color}

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[label for _, label, _ in peak_params],
    vertical_spacing=0.3,
    horizontal_spacing=0.15,
)
for idx, (col, _, ytitle) in enumerate(peak_params):
    row_idx, col_idx = divmod(idx, n_cols)
    for sample in ["BSA", "BSA+TCEP"]:
        df_s = peak_df[peak_df["sample"] == sample].sort_values("pH")
        df_low = df_s[df_s["pH"] <= 3]
        df_high = df_s[df_s["pH"] >= 7.5]
        color = sample_colors[sample]

        # Solid lines for each measured pH range
        for seg_idx, df_seg in enumerate([df_low, df_high]):
            fig.add_trace(
                go.Scatter(
                    x=df_seg["pH"],
                    y=df_seg[col],
                    mode="lines+markers",
                    name=sample,
                    legendgroup=sample,
                    showlegend=(idx == 0 and seg_idx == 0),
                    line=dict(color=color),
                ),
                row=row_idx + 1,
                col=col_idx + 1,
            )

        # Dashed connector bridging the unmeasured pH gap (3–7.5)
        fig.add_trace(
            go.Scatter(
                x=[df_low["pH"].iloc[-1], df_high["pH"].iloc[0]],
                y=[df_low[col].iloc[-1], df_high[col].iloc[0]],
                mode="lines",
                name=sample,
                legendgroup=sample,
                showlegend=False,
                line=dict(color=color, dash="dash"),
            ),
            row=row_idx + 1,
            col=col_idx + 1,
        )
        fig.update_yaxes(title_text=ytitle, row=row_idx + 1, col=col_idx + 1)

fig.update_layout(legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1))
fig.update_xaxes(title_text="pH")
apc.plotly.style_plot(fig, monospaced_axes="all")
apc.plotly.set_figure_dimensions(fig, size="full_wide")
fig.update_layout(height=500)
fig.show()

fig.write_image(plot_dir_png / "peaks_vs_pH.png", scale=3)
fig.write_html(plot_dir_htmls / "peaks_vs_pH.html")


## Write experiment -> raw data mapping table

The dictionary below contains every figure generated above as keys. For experiments where SEC data is present, the corresponding SEC data is added as a value.

In [ ]:
figure_dict = {"Figure 1 (BSA, 10 mg/ml)": "BSA",
               "Supporting figure (FAF-BSA, 10 mg/ml)": None,
               "Figure 2 (DNAse I, 12.5 mg/ml)": "DNase",
               "Figure 3 (Cytochrome C, 10 mg/ml)": "CytC",
               "Supporting Figure (BSA, 3 mg/ml)": None,
               "Supporting Figure (FAF-BSA, 3 mg/ml)": None,
               "Supporting Figure (Trypsin, 3 mg/ml)": None,
               "Supporting Figure (Trypsin, 10 mg/ml)": None,
               "Supporting Figure (Hemoglobin, 3 mg/ml)": None,
               "Supporting Figure (Hemoglobin, 10 mg/ml)": "Hemoglobin",
               "Supporting Figure (Cytochrome C, 3 mg/ml)": None,
               "Supporting Figure (BSA 10 mg/ml, replicate)": None,
               "Supporting Figure (BSA 3 mg/ml, replicate)": None,
               "Supporting Figure (IgG 8.6 mg/ml)": "IgG",
               "Supporting Figure (Buffers)": None,
               "Figure 4 (BSA pH perturbation)": None,
               "Figure 5 (BSA+TCEP pH perturbation)": None
            }


In [ ]:
def get_SEC_df(protein_name):
    """Helper function to get the SEC dataframe for a given protein name."""
    dir_path = data_dir / "SEC" / f"{protein_name}_raw_data"
    file_dirs = [f.relative_to(repo_root) for f in dir_path.glob("*.csv") if f.is_file()]
    df_SEC = pd.DataFrame(data={"Filename": file_dirs})
    return df_SEC

with open(outputs_dir / "experiment-sample-map.md", "w") as f:
    for fig_id, df in zip(figure_dict.keys(), df_sample_tables, strict=True):
        # add repo-relative path to filenames
        if "pH" in fig_id:
            df["Protein filename"] = (
                raman_ph_perturbation_dir.relative_to(repo_root) / df["Protein filename"]
            )
            if df["Buffer filename"].notnull().any():
                df["Buffer filename"] = (
                    raman_ph_perturbation_dir.relative_to(repo_root) / df["Buffer filename"]
                )
        else:
            df["Protein filename"] = (
                raman_chem_perturbation_dir.relative_to(repo_root) / df["Protein filename"]
            )
            if df["Buffer filename"].notnull().any():
                df["Buffer filename"] = (
                    raman_chem_perturbation_dir.relative_to(repo_root) / df["Buffer filename"]
                )
        df_md = df.to_markdown(index=False)
        f.write(f"## {fig_id}\n\n")
        f.write(df_md)

        if figure_dict[fig_id] is not None:
            df_SEC = get_SEC_df(figure_dict[fig_id])
            f.write(f"\n\n#### SEC data for {figure_dict[fig_id]}\n\n")
            f.write(df_SEC.to_markdown(index=False))
        f.write("\n\n---\n\n")

### BSA melt curve from raw spectra (same pipeline as pH analysis)
Spectra from all 3 sites per well are averaged before processing. Wells G11 (4 °C, 20 mg/ml) and F10 (70 °C, low signal and artifacts) are excluded.

In [ ]:
bsa_heat_raw_dir = Path(
    "/Users/ilya/Code/2026-matt-ppp/PPP_BSA_Heat_003_SS+GB_analysis/mfc_ppp_bsa_heat_003_ss96"
)
bsa_heat_plate_map = pd.read_csv(
    bsa_heat_raw_dir / "bsa_heat_003_ss_plate_map_96well_stainless_steel_2026-08-27.csv",
    comment="#",
)
# Excluded wells: G11 (4 °C, 20 mg/ml), F10 (70 °C, low signal and artifacts)
bsa_heat_plate_map = bsa_heat_plate_map[~bsa_heat_plate_map["well"].isin(["G11", "F10"])]
buffer_wells = bsa_heat_plate_map.loc[bsa_heat_plate_map["sample_name"] == "tris buff", "well"]
temp_wells = bsa_heat_plate_map[bsa_heat_plate_map["sample_name"].str.fullmatch(r"\d+")].copy()
temp_wells["temperature"] = temp_wells["sample_name"].astype(int)
temp_wells = temp_wells.sort_values("temperature")
temps_to_plot = temp_wells["temperature"].tolist()
temps_to_show_on_colorbar = [4, 66, 80]
CH2_wavenumber = 1450  # CH₂ deformation peak, used for normalization of heat data
water_band = (3300, 3500)  # O-H stretch region, used to scale buffer before subtraction

heat_data_list, _ = collect_spectra_from_dir(bsa_heat_raw_dir, "instantraman")


def mean_raw_spectrum(wells, col="Intensity (dark-subtracted)"):
    """Average the raw spectra of all sites in the given wells (before any processing)."""
    dfs = [df for f, df in heat_data_list if f.split("-Site_")[0] in set(wells)]
    wavenumbers = dfs[0]["Wavenumber"].values
    assert all(np.allclose(df["Wavenumber"].values, wavenumbers) for df in dfs)
    return rp.Spectrum(np.mean([df[col].values for df in dfs], axis=0), wavenumbers)


def water_band_intensity(spectrum):
    in_band = (spectrum.spectral_axis >= water_band[0]) & (spectrum.spectral_axis <= water_band[1])
    return spectrum.spectral_data[in_band].mean()


# Subtract mean of all buffer spectra from each site-averaged sample, then apply the pH pipeline.
# Total signal drops 2-4x at >=70 °C (gelled/turbid samples), so the buffer is scaled to match each
# sample's water O-H stretch intensity; otherwise water is over-subtracted.
buffer_spectrum = mean_raw_spectrum(buffer_wells)
heat_spectra = {}
for well, temp in zip(temp_wells["well"], temp_wells["temperature"], strict=True):
    spectrum = mean_raw_spectrum([well])
    buffer_scale = water_band_intensity(spectrum) / water_band_intensity(buffer_spectrum)
    spectrum.spectral_data = spectrum.spectral_data - buffer_scale * buffer_spectrum.spectral_data
    heat_spectra[f"BSA {temp} °C"] = rp_pipeline.apply(spectrum)

fig = plot_spectra(
    heat_spectra,
    data_dir=bsa_heat_raw_dir,
    height=ph_fig_height,
    width=ph_fig_width,
    xaxis_title="Raman shift (cm⁻¹)",
    yaxis_title="Intensity (norm. to Phe)",
    title="BSA heat perturbation",
)
normalize_to_peak(fig, Phe_wavenumber)  # CH2_wavenumber)

gradient = ph_gradient.reverse().resample_as_palette(len(fig.data))
for i, trace in enumerate(fig.data):
    trace.line.color = gradient[i]

# Temperature color bar. Trace colors are evenly spaced by index (not by temperature, since 4 °C
# is far from the rest), so the colorbar is indexed the same way and labeled with temperatures
colorbar_trace = go.Scatter(
    x=[None],
    y=[None],
    mode="markers",
    marker=dict(
        colorscale=[[i / (len(gradient) - 1), color] for i, color in enumerate(gradient)],
        cmin=0,
        cmax=len(temps_to_plot) - 1,
        colorbar=dict(
            title="Temperature (°C)",
            x=1.0,
            y=1.05,
            xanchor="left",
            yanchor="bottom",
            orientation="h",
            thickness=5,
            lenmode="pixels",
            len=200,
            title_side="top",
            tickvals=[temps_to_plot.index(t) for t in temps_to_show_on_colorbar],
            ticktext=[str(t) for t in temps_to_show_on_colorbar],
        ),
        color=[0, len(temps_to_plot) - 1],
        showscale=True,
        size=0.1,
    ),
    showlegend=False,
    hoverinfo="none",
)
fig.add_trace(colorbar_trace)

fig.update_layout(margin=dict(t=100))
apc.plotly.style_plot(fig, monospaced_axes="all")
apc.plotly.set_figure_dimensions(fig, size="full_wide")
fig.show()

fig.write_image(plot_dir_png / "BSA_heat_spectra.png", scale=3)
fig.write_html(plot_dir_htmls / "BSA_heat_spectra.html")

### Plot Amide I peak features as a function of temperature (from raw spectra)

In [ ]:
amide_I_baseline_anchors = (1626, 1720)  # cm⁻¹, ends of the Amide I region
anchor_window = 5  # cm⁻¹, half-width of the window around each anchor used for the baseline fit


def subtract_linear_baseline(spectral_axis, spectral_data_row, anchors, window):
    """Subtract a line fit to the points within ±window of each anchor wavenumber."""
    in_anchors = np.any([np.abs(spectral_axis - a) <= window for a in anchors], axis=0)
    slope, intercept = np.polyfit(spectral_axis[in_anchors], spectral_data_row[in_anchors], 1)
    return spectral_data_row - (slope * spectral_axis + intercept)


heat_peak_records = []
for name, spectrum in heat_spectra.items():
    ax, row = spectrum.spectral_axis, spectrum.spectral_data
    # Some spectra have an elevated baseline under Amide I; flatten it before fitting the peak.
    # The 1,450 cm⁻¹ intensity used for normalization is taken from the uncorrected spectrum.
    row_amide = subtract_linear_baseline(ax, row, amide_I_baseline_anchors, anchor_window)
    center_1657, fwhm_1657 = fit_gaussian_peak(ax, row_amide, 1657)
    heat_peak_records.append(
        {
            "temperature": int(name.split()[1]),
            "i1657": (
                intensity_at(ax, row_amide, center_1657) / peak_max(ax, row, CH2_wavenumber)
            ),
            "i1657_center": center_1657,
            "i1657_fwhm": fwhm_1657,
        }
    )
heat_peak_df = pd.DataFrame(heat_peak_records).sort_values("temperature")

heat_peak_params = peak_params[:3]  # Amide I peak, center, FWHM

# Broken x axis: each panel is a narrow axis for the 4 °C point and a wide axis for 60-80 °C,
# separated by spacer columns between panels
low_xrange, high_xrange = [0, 8], [58, 82]
low_width, high_width, spacer_width = 0.04, 0.22, 0.09
n_params = len(heat_peak_params)
column_widths = [low_width, high_width, spacer_width] * n_params
column_widths = column_widths[:-1]  # no spacer after the last panel
specs = ([{}, {}, None] * n_params)[:-1]
fig = make_subplots(
    rows=1,
    cols=len(column_widths),
    column_widths=column_widths,
    specs=[specs],
    horizontal_spacing=0.01,
)
df_low = heat_peak_df[heat_peak_df["temperature"] <= low_xrange[1]]
df_high = heat_peak_df[heat_peak_df["temperature"] >= high_xrange[0]]
for idx, (col, label, ytitle) in enumerate(heat_peak_params):
    low_col, high_col = 3 * idx + 1, 3 * idx + 2
    for df_seg, plot_col in [(df_low, low_col), (df_high, high_col)]:
        fig.add_trace(
            go.Scatter(
                x=df_seg["temperature"],
                y=df_seg[col],
                mode="lines+markers",
                name="BSA",
                legendgroup="BSA",
                showlegend=(idx == 0 and plot_col == low_col),
                line=dict(color=BSA_color),
            ),
            row=1,
            col=plot_col,
        )

    # Dashed connector across the break, split at the break so it looks continuous
    x_low, y_low = df_low["temperature"].iloc[-1], df_low[col].iloc[-1]
    x_high, y_high = df_high["temperature"].iloc[0], df_high[col].iloc[0]
    d_low = (low_xrange[1] - x_low) / np.diff(low_xrange)[0] * low_width
    d_high = (x_high - high_xrange[0]) / np.diff(high_xrange)[0] * high_width
    y_break = y_low + (y_high - y_low) * d_low / (d_low + d_high)
    for xs, ys, plot_col in [
        ([x_low, low_xrange[1]], [y_low, y_break], low_col),
        ([high_xrange[0], x_high], [y_break, y_high], high_col),
    ]:
        fig.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode="lines",
                showlegend=False,
                hoverinfo="skip",
                line=dict(color=BSA_color, dash="dash"),
            ),
            row=1,
            col=plot_col,
        )

    fig.update_xaxes(range=low_xrange, tickvals=[4], row=1, col=low_col)
    fig.update_xaxes(range=high_xrange, title_text="Temperature (°C)", row=1, col=high_col)
    fig.update_yaxes(title_text=ytitle, row=1, col=low_col)
    low_yaxis = fig.get_subplot(1, low_col).yaxis.plotly_name.replace("axis", "")
    fig.update_yaxes(matches=low_yaxis, showticklabels=False, row=1, col=high_col)

    # Panel title centered over both axes
    low_domain = fig.get_subplot(1, low_col).xaxis.domain
    high_domain = fig.get_subplot(1, high_col).xaxis.domain
    fig.add_annotation(
        text=label,
        x=(low_domain[0] + high_domain[1]) / 2,
        y=1.0,
        xref="paper",
        yref="paper",
        xanchor="center",
        yanchor="bottom",
        showarrow=False,
        font=dict(size=16),
    )

fig.update_layout(legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1))
apc.plotly.style_plot(fig, monospaced_axes="all")
apc.plotly.set_figure_dimensions(fig, size="full_wide")
fig.update_layout(height=300)

# Hide the y axis line of each high-temperature axis and draw break marks on the x axis
for idx in range(n_params):
    high_col = 3 * idx + 2
    fig.update_yaxes(showline=False, ticks="", row=1, col=high_col)
    for col_, x_edge in [(3 * idx + 1, 1), (high_col, 0)]:
        x_paper = fig.get_subplot(1, col_).xaxis.domain[x_edge]
        fig.add_shape(
            type="line",
            x0=x_paper - 0.004,
            x1=x_paper + 0.004,
            y0=-0.04,
            y1=0.04,
            xref="paper",
            yref="paper",
            line=dict(color="black", width=1),
        )
fig.show()

fig.write_image(plot_dir_png / "amide_I_vs_temperature.png", scale=3)
fig.write_html(plot_dir_htmls / "amide_I_vs_temperature.html")